In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
HERE = %pwd
sys.path.append(os.path.dirname(HERE))

%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
    
import numpy as np
import pandas as pd
import copy
import pickle
import time
import collections
from tqdm import tqdm

In [2]:
from src import utils
rng = utils.set_seed()

dir_parent = utils.dir_parent
version_exp = utils.version_exp
dir_workspace = f"{dir_parent}/research/TFCSR"

In [3]:
dir_load = f"{dir_parent}/received_data/amazon23"

# parameters
n_positive = 2
n_candidate = 50 - n_positive
n_user = 500

In [4]:
import gzip
import json
from tqdm import tqdm


def _records(path_records):
    g = gzip.open(path_records, 'rb')
    dict_ = {}
    idx = 0
    for l in tqdm(g):
        d_ = json.loads(l)
    
        try:
            user = d_['user_id']
            item = d_['parent_asin']
            rating = d_['rating']
            
            time = d_["timestamp"]
            #text = utils.to_text(d_["text"])
    
            if d_['verified_purchase'] is True:
                l = [item, rating, time]
                if user in dict_.keys():
                    dict_[user].append(l)
                else:
                    dict_[user] = [l]
        except:
            pass
    
    # transform dict to pandas.DataFrame
    def _reshape(user):
        df = pd.DataFrame(dict_[user], columns=['itemID', 'rating', 'time'])
        df.insert(0, "userID", user)
        return df
    
    df_ = pd.concat([_reshape(user) for user in tqdm(dict_.keys())])
    df_.reset_index(inplace=True, drop=True)
    
    # sort chronological order
    df_ = df_.sort_values(by="time", ascending=False)  
    
    df_["userID"] = [f"U_{i}" for i in df_["userID"].values]
    df_["itemID"] = [f"I_{i}" for i in df_["itemID"].values]
    
    # delete dupicated items
    df_records = df_.drop_duplicates(subset=["userID", "itemID"], keep='first').reset_index(drop=True)
    return df_records


def _items(path_items):
    g = gzip.open(path_items, 'rb')
    dict_ = {}
    idx = 0
    for l in tqdm(g):
        d_ = json.loads(l)
        item = d_["parent_asin"]
        
        # title
        try:
            title = utils.to_text(d_['title'])
        except:
            title = ""
    
        ## long word title tends to be weird, so skip it
        ## short word title tends to be weird, so skip it
        if (utils.compute_token(title) > 200) or (len(set(title)) <= 3):
            title = ""
        
        # category
        try:
            cat = d_['categories']
            cat = [utils.to_text(c.replace("&amp;", "&")) for c in cat[1:]]
            cat = [c for c in cat if len(c) <= 50]
            categories = ", ".join(cat)
        except:
            categories = ""
        
        # description
        try:
            description = utils.to_text(", ".join(d_['description']))
    
            ## remove short description
            if len(set(description)) <= 10:
                description = ""
    
            ## delete items whose description were longer than 300 tokens
            if utils.compute_token(description) >= 300:
                description = ""
        except:
            description = ""
    
        l = [title, categories, description]
        if item in dict_.keys():
            # update longer description
            if len(description) > len(dict_[item][2]):
                dict_[item] = l
        else:
            dict_[item] = l
    
    df_ = pd.DataFrame(dict_, index=["title", "category", "description"]).T
    df_ = df_.dropna()
    df_.index = [f"I_{i}" for i in df_.index]
    
    df_items = df_[df_["title"] != ""]   
    return df_items

def _remove_un_used_items(df_records, df_items):
    # items registered in item master
    items_master = set(df_items.index.values)

    # items that are not super long tail
    s = df_records["itemID"].value_counts()
    items_freq = set(s[s>1].index.values)
    items_master = items_master.intersection(items_freq)
    
    # restrict transaction records whose rows are registered in items_master.
    s = df_records['itemID'].apply(lambda s : s in items_master)
    df_r = df_records[s]

    # restrict item master to items registered in restricted transaction records
    df_i = df_items.loc[df_r['itemID'].unique()]
    return df_r, df_i

In [5]:
data_names = [
    "CDs_and_Vinyl", "Movies_and_TV", "Toys_and_Games", "Sports_and_Outdoors"
]
N_icl = [1, 3, 5, 10]

for data_name in data_names:
    print(data_name)
    dir_data = f"{dir_workspace}/preprocessed_data/{version_exp}/ARD_{data_name}"
    os.makedirs(dir_data, exist_ok=True)

    # create preprocessed data
    try:
        for n_icl in N_icl:
            user_type = f"{n_icl}-sample"
            with open(f"{dir_data}/records_{user_type}.pickle", 'rb') as f:
                d_data = pickle.load(f)
            user = list(d_data.keys())[-1]
            print(user_type, d_data[user])
        
        for item_type in ["history", "candidates"]:
            df_i = pd.read_csv(f"{dir_data}/items_{item_type}.csv", index_col=0)
            display(df_i.head())        
            print(item_type, len(df_i))
    except:
        # load full preprocessed data
        try:
            with open(f"{dir_data}/records.pickle", 'rb') as f:
                df_records = pd.DataFrame(pickle.load(f)).T.infer_objects(copy=False)
            with open(f"{dir_data}/items.pickle", 'rb') as f:
                df_items = pd.DataFrame(pickle.load(f)).T.infer_objects(copy=False)
        except:
            path_records = f"{dir_load}/{data_name}.jsonl.gz"
            df_records = _records(path_records)
            
            path_items = f"{dir_load}/meta_{data_name}.jsonl.gz"
            df_items = _items(path_items)
        
            df_records, df_items = _remove_un_used_items(df_records, df_items)

            with open(f"{dir_data}/records.pickle", 'wb') as f:
                pickle.dump(df_records.T.to_dict(), f)
            with open(f"{dir_data}/items.pickle", 'wb') as f:
                pickle.dump(df_items.T.to_dict(), f)                

        # remove NaN description
        df_items = df_items[df_items["description"] != ""]
        df_records, df_items = _remove_un_used_items(df_records, df_items)
        
        # ================ prepare ================ 
        # frequent items up to rank 10000
        s = df_records["itemID"].value_counts()
        s = s.sort_values(ascending=False).iloc[:1000]
        set_items = set(s.index)
        
        gb = df_records.groupby("userID")
        users = df_records["userID"].unique()
        k = -1*n_positive
        
        def _items_user(user, gb):
            df_ = gb.get_group(user).sort_values(by="time", ascending=True)
            df_ = df_[df_["rating"] > 3]
            items_user = df_["itemID"].values    
            return items_user
        
        def _candidate_items(items_user, set_items, k, n_candidate):
            items_pos = items_user[k:]  # latest k items
            items_neg = rng.choice(list(set_items - set(items_user)), size=n_candidate, replace=False)      
            d_ = {
                "candidates_positive" : ", ".join(items_pos),
                "candidates_negative" : ", ".join(items_neg)
            }
            return copy.deepcopy(d_)
        
        # ================ User history ================ 
        dd_data = dict()
        for n_icl in N_icl:
            # shuffle
            users = rng.choice(users, size=len(users), replace=False)
            
            d_data = dict()
            idx = 0
            for user in tqdm(users, desc=f"{n_icl:2}-sample"):
                items_user = _items_user(user, gb)
                
                if len(items_user) >= n_icl + n_positive:
                    try:
                        # candidate items
                        d_ = _candidate_items(items_user, set_items, k, n_candidate)
                        
                        # user history (except latest k items)
                        d_["history"] = {i+1 : item for i,item in enumerate(items_user[:k][::-1][:n_icl])} 
                        d_data[user] = copy.deepcopy(d_)
                        idx += 1
                    except:
                        pass
                
                if idx == n_user:
                    print(user, d_data[user])
                    break

            dd_data[f"{n_icl}-sample"] = copy.deepcopy(d_data)
    
        # ================ Save ================ 
        d_item = dict()
        for user_type, d_data in dd_data.items():
            with open(f"{dir_data}/records_{user_type}.pickle", 'wb') as f:
                pickle.dump(d_data, f)
            
            # items that used in the main experiments
            try:
                items_history = np.unique(np.concatenate([list(d["history"].values()) for d in d_data.values()]))
            except:
                items_history = []
            
            items_pos = np.unique(np.concatenate([d["candidates_positive"].split(", ") for d in d_data.values()]))
            items_neg = np.unique(np.concatenate([d["candidates_negative"].split(", ") for d in d_data.values()]))
            items_candidates = np.unique(np.concatenate([items_pos, items_neg]))
                    
            d_item[user_type] = {
                "history" : items_history,
                "candidates" : items_candidates
            }
            
        for item_type in ["history", "candidates"]:
            items_ = np.unique(np.concatenate([d[item_type] for d in d_item.values()]))
            df_i = df_items.loc[items_]
            df_i.to_csv(f"{dir_data}/items_{item_type}.csv")

CDs_and_Vinyl
1-sample {'candidates_positive': 'I_B000002LSV, I_B000001EYI', 'candidates_negative': 'I_B00BY8DIT2, I_B00914JWSW, I_B0168KQSYM, I_B078SKX6ZV, I_B008O9V4C2, I_B079PDLQV5, I_B004ZN9T00, I_B074YCSNT4, I_B004KBSQBA, I_B000E6EJAW, I_B01NH3ABQH, I_B008FSCNTK, I_B006ZCWU5K, I_B012DXW5DQ, I_B01GGJV5VQ, I_B00006L736, I_B006ZZAMWK, I_B00T3YBPLC, I_B000002GCB, I_B0039TD7PY, I_B019CL2SF2, I_B01L2ZS87S, I_B00LTM1D5O, I_B00000K3WY, I_B005MQNDVK, I_B000002JUC, I_B003V6L94Q, I_B00C81AVNU, I_B001U9BRNE, I_B00005NKKN, I_B079FLRBVM, I_B000089RV6, I_B001AQTWF2, I_B00OMKEK64, I_B00N0T38LG, I_B000WAEJZ0, I_B0007OY474, I_B00631VJIC, I_B00942S4OY, I_B00A1EOBJQ, I_B08HGB71RT, I_B0002YJ2DU, I_B000002VMD, I_B01I5AI96S, I_B00000136Z, I_B0884BK38T, I_B000063CNC, I_B0086449YA', 'history': {1: 'I_B000003A60'}}
3-sample {'candidates_positive': 'I_B00SSJIAXE, I_B07WP752DH', 'candidates_negative': 'I_B004EBT5CU, I_B00MA151ZQ, I_B008K9SG9K, I_B07WHMPTBH, I_B00GRHVIGU, I_B00DRIMVK4, I_B000065UFD, I_B00BQ1D

,title,category,description
I_1559617233,Brainwave Suite,"New Age, Meditation",We all experience many states of consciousness...
I_1568556888,Bobby Darin: Mack is Back,"Pop, Vocal Pop","Product Description, Now for the first time, v..."
I_1573300411,LIVE & LOUD,"International Music, Europe, British Isles, Br...",This is the first live home video in Ozzy's il...
I_1573306983,John Denver: The WILDlife Concert,"Pop, Singer-Songwriters","Celebrate John Denver's greatest hits with ""Th..."
I_1582703272,Body-Field Sound Healing,Today's Deals deprecated,The first-of-its-kind audio experience in whic...


history 8196


,title,category,description
I_0005164885,Christmas Eve and Other Stories,"Holiday & Wedding, Christmas","Product Description, A new sound for the seaso..."
I_0739003755,Music for Little Mozarts Sets for Lesson and D...,"Broadway & Vocalists, Musicals",Two compact disc recordings (sold as a two-dis...
I_1569383448,Bernadette Peters in Concert,"Pop, Vocal Pop","From the Back Cover, ""A diva who effuses sweet..."
I_1573306983,John Denver: The WILDlife Concert,"Pop, Singer-Songwriters","Celebrate John Denver's greatest hits with ""Th..."
I_6302320690,Neil Diamond: Love At The Greek VHS 1976,"Pop, Adult Contemporary",Love at the Greek [VHS]


candidates 4238
Movies_and_TV
1-sample {'candidates_positive': 'I_B00HW3EI3I, I_B094SRC3VY', 'candidates_negative': 'I_B00BCRRA8U, I_B00BUWD7Y8, I_B005OCFGTO, I_B00KHD5FK0, I_B004DK5CW4, I_B00X797NJW, I_B07NFBPT41, I_B00NCDVVLY, I_B006H5120E, I_B0000633U2, I_B000BPL2GK, I_B0087F7VSY, I_B00PGV9X5G, I_B0045HCJRG, I_B00KQTGWPC, I_B00HUFALRK, I_B00O4ZC57I, I_B00005JLHX, I_B01JAQEQOK, I_B005LJK3AC, I_B006O5Y1FK, I_B00AMSM9CW, I_B0713MM9DT, I_B00K2CHVJ4, I_B00SI7GDBM, I_B00E9ZAT4Y, I_B00APE1NZW, I_B004SEUJ82, I_B008JPZUYE, I_B00CNW9ZI6, I_B001NFNFMQ, I_B01MTD0LE5, I_B00AATV046, I_B00KKGS1KK, I_B00B1LN8WY, I_B017V2IG6O, I_0790731533, I_B000WC38BO, I_B00XQ13ZSO, I_B00WAJ8QCS, I_B01LTHXYXM, I_B009EEVSTK, I_B01JTQ3LFG, I_B00B4804KS, I_B002SAMMEC, I_B000BX5X5I, I_B00WG1DDFU, I_B00DYQ1G78', 'history': {1: 'I_B00OPNQ66K'}}
3-sample {'candidates_positive': 'I_B072JLDDMT, I_B00DPH7V8E', 'candidates_negative': 'I_B073ZQ3DZQ, I_B00EV4EUT8, I_B00UI5CU6Y, I_B00DKS3QZU, I_B07S86K91M, I_B01C6JFGFQ, I_B00NC

...

candidates 4276
Sports_and_Outdoors
1-sample {'candidates_positive': 'I_B00MPE8GFK, I_B0744GWMMQ', 'candidates_negative': 'I_B0BS54CFF5, I_B017UMB0OA, I_B077X1SQY9, I_B00404R4NI, I_B00D3WV3VC, I_B07C2V3J7C, I_B00NPZ73QQ, I_B075HN78M9, I_B0BXHLWCR5, I_B07KG4179P, I_B01A04FLWW, I_B0C4MNRN7R, I_B004XADAQO, I_B09Q59D5SC, I_B07NYZS5VG, I_B07NVF3Q1R, I_B0C45XP1MM, I_B003V57NUQ, I_B0BCL6KCF8, I_B09TGBGM8D, I_B00701H384, I_B09JC1ZC2Y, I_B0B4JNZ875, I_B005AG4O42, I_B09GXBKQKN, I_B013FIVU0C, I_B08WQV6YPR, I_B07XBTG8QS, I_B0088ZVLAM, I_B0C48Q7YDS, I_B08225SJL5, I_B01LXGBP22, I_B07GBFYHNM, I_B001C1UGVO, I_B002VQ9PU2, I_B0BF1KKXMJ, I_B00P7MYS6I, I_B00GN85A3A, I_B09M4DCXH6, I_B00GN856F2, I_B07NWQSV2Y, I_B07R6NX9YF, I_B07ZSDK651, I_B0B5GP8JPB, I_B006JH1HDW, I_B001ASJDWW, I_B00873J0YY, I_B0BYDCRCW2', 'history': {1: 'I_B07BVBX3V2'}}
3-sample {'candidates_positive': 'I_B07G7TR8T4, I_B074PCX7N7', 'candidates_negative': 'I_B07DYB9G79, I_B07F95GK37, I_B07D6M2152, I_B0C78RCGWB, I_B00F9F6OVK, I_B000E3D2VM, I

,title,category,description
I_1609968417,Carson Dellosa Colorful Owls Shape Stickers (1...,"Fan Shop, Toys & Game Room",These colorful owl die-cut shape stickers are ...
I_1944788921,GymPad Mini Workout Journal - The Small Stylis...,"Exercise & Fitness, Accessories, Fitness Planners","Available in 4 different colours, which one wi..."
I_2094869245,5 LED Bicycle Rear Tail Red Bike Torch Laser B...,"Sports, Cycling, Accessories, Lights & Reflect...",This newly-designed Laser tail light can emit ...
I_7245456313,Black Mountain Products Resistance Band Set wi...,"Exercise & Fitness, Strength Training Equipmen...",Black Mountain Products (B.M.P.) resistance ba...
I_7500670133,Titanium Tornado Baseball Necklace Orange Blac...,"Sports, Team Sports, Baseball, Accessories",Materials emit energy that is effective in con...


history 8108


,title,category,description
I_1469393182,"Chicago Bears 2023 12"" x 12"" Team Wall Calendar","Fan Shop, Office Products, Calendars & Planners",Start the season off right with this team 2023...
I_7245456313,Black Mountain Products Resistance Band Set wi...,"Exercise & Fitness, Strength Training Equipmen...",Black Mountain Products (B.M.P.) resistance ba...
I_B0000ANBEZ,Coleman 3000000455 Latern Pump Kit,"Outdoor Recreation, Camping & Hiking, Camp Kit...",Coleman Liquid Fuel Lantern and Camp Stove Pum...
I_B0000AQQX3,"Fulton XP10 0101 Swivel Trailer Tongue Jack, 1...","Sports, Boating & Sailing, Boating, Boat Trail...",Fulton XP10 0101 Trailer Jack is designed for ...
I_B0000AUT1J,Coleman 3000000454 Filler Cap Lantern,"Outdoor Recreation, Camping & Hiking, Lights &...",Coleman 3000000454 filler cap. Camping lights ...


candidates 4326
